In [23]:
## Take monthly grib ERA5 data on the native grid and interpolate to lower resolution, pressure levels
## For individual daily, monthly and climatology of daily, monthly

In [24]:
import xarray as xr
import numpy as np
import pandas as pd

import cfgrib as cfgrib
import xesmf as xe

In [25]:
from dask.distributed import Client
from ncar_jobqueue import NCARCluster

In [26]:
cluster = NCARCluster(project='P93300042',interface='ext',memory='30GB',cores=4,processes=1)
cluster
cluster.scale(jobs=4)
client = Client(cluster)
client

/glade/u/apps/opt/conda/envs/npl-2025a/lib/python3.12/site-packages/dask_jobqueue/core.py:266: FutureWarning: job_extra has been renamed to job_extra_directives. You are still using it (even if only set to []; please also check config files). If you did not set job_extra_directives yet, job_extra will be respected for now, but it will be removed in a future release. If you already set job_extra_directives, job_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/glade/u/apps/opt/conda/envs/npl-2025a/lib/python3.12/site-packages/dask_jobqueue/core.py:285: FutureWarning: env_extra has been renamed to job_script_prologue. You are still using it (even if only set to []; please also check config files). If you did not set job_script_prologue yet, env_extra will be respected for now, but it will be removed in a future release. If you already set job_script_prologue, env_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/glade/u/apps/opt/con

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/rneale/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/rneale/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.115:41051,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/rneale/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


## Read in one grib file at a time then
-Strip Uneeded data



In [27]:
# Read in grid? data
#file_in ='/glade/derecho/scratch/rneale/ERA5/download/dtdt_param/dtdt_param_1979_01_ytest_era5_modelevs.nc'

#ds_in = xr.open_dataset(file_in, chunks={'time': 10})
#ds_in





In [28]:
# Read in grid? data
file_in ='/glade/derecho/scratch/rneale/ERA5/download/dtdt_param/dtdt_param_1979_01_ytest_era5_modelevs.grib'
file_ps = '/glade/derecho/scratch/rneale/ERA5/download/sp/sp_1979_01_ytest_era5_modelevs.grib'

varname = 'avg_ttpm'

ds_in = xr.open_dataset(file_in, chunks={'time': 10},engine='cfgrib')
ds_sp = xr.open_dataset(file_ps, chunks={'time': 10},engine='cfgrib')

#ds_in = cfgrib.open_datasets(file_in)

ds_in

Ignoring index file '/glade/derecho/scratch/rneale/ERA5/download/sp/sp_1979_01_ytest_era5_modelevs.grib.5b7b6.idx' incompatible with GRIB file


<xarray.Dataset> Size: 12GB
Dimensions:     (time: 62, hybrid: 88, values: 542080)
Coordinates:
  * time        (time) datetime64[ns] 496B 1979-01-01T06:00:00 ... 1979-01-31...
    step        timedelta64[ns] 8B ...
  * hybrid      (hybrid) float64 704B 50.0 51.0 52.0 53.0 ... 135.0 136.0 137.0
    latitude    (values) float64 4MB dask.array<chunksize=(542080,), meta=np.ndarray>
    longitude   (values) float64 4MB dask.array<chunksize=(542080,), meta=np.ndarray>
    valid_time  (time) datetime64[ns] 496B dask.array<chunksize=(10,), meta=np.ndarray>
Dimensions without coordinates: values
Data variables:
    avg_ttpm    (time, hybrid, values) float32 12GB dask.array<chunksize=(10, 88, 542080), meta=np.ndarray>
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-05-23T15:27 GRIB to CDM+CF via cfgrib-0.9.1...

In [29]:
''' Prepare Target Lat-Lon Grid '''

# Target grid (0.25° resolution)
lat_out = np.arange(-90, 90.1, 1.0)
lon_out = np.arange(0, 360, 1.0)

grid_out = xr.Dataset({
    "lat": (["lat"], lat_out),
    "lon": (["lon"], lon_out),
})





# Get unique lat/lon and their 2D shape
lat_vals = ds_in.latitude.values
lon_vals = ds_in.longitude.values
n_total = len(lat_vals)

# Guess 2D shape (e.g., 721x1440 for 0.25°)
nlat = len(np.unique(lat_vals))
nlon = n_total // nlat

# Sort and reshape
lat2d = lat_vals.reshape(nlat, nlon)
lon2d = lon_vals.reshape(nlat, nlon)

# Build 2D coordinate system
lat_coords = np.unique(lat2d[:, 0])
lon_coords = np.unique(lon2d[0, :])

# Select the variable to regrid

data = ds_in[varname]  # shape: (time, level, rgrid)






''' OLD 2D Approach '''

# Extract data
#lat = ds_in.latitude.values
#lon = ds_in.longitude.values

# Number of unique latitudes
#unique_lats = np.unique(lat)
#unique_lons = np.unique(lon)

# Check if the data can form a 2D grid
#print(len(lat), len(unique_lats), len(unique_lons))

' OLD 2D Approach '

In [30]:

''' INTERPOLATE TO REGULAR LAT/LON GRID (2D) '''

# Create a regular grid
lat_out = np.arange(-90, 90.1, 1.)
lon_out = np.arange(0, 360, 1.)
grid_out = xr.Dataset({
    'lat': (['lat'], lat_out),
    'lon': (['lon'], lon_out)
})

# Prepare the input grid
grid_in = xr.Dataset({
    'lat': (['rgrid'], lat),
    'lon': (['rgrid'], lon),
})

# Select a variable to regrid (e.g., temperature)
varname = list(ds_in.data_vars)[0]
data = ds_in[var_name]

# xESMF wants lat/lon as 2D, so we need to reshape
data_2d = data.isel(time=0, isobaricInhPa=0).values.reshape(nlat, nlon)

data_in = xr.DataArray(data_2d, dims=["lat", "lon"],
                       coords={"lat": (["lat"], unique_lats),
                               "lon": (["lon"], np.linspace(0, 360, nlon, endpoint=False))})

# Regrid
regridder = xe.Regridder(data_in, grid_out, method='bilinear', periodic=True)
data_out = regridder(data_in)
data_out

ValueError: Dimensions {'isobaricInhPa'} do not exist. Expected one or more of ('time', 'hybrid', 'values')

In [ ]:
''' INTERPOLATE TO REGULAR LAT/LON GRID (3D) '''


# Empty list to hold regridded slices
output_slices = []

# Regridder can be reused
dummy_input = xr.DataArray(
    np.zeros((nlat, nlon)),
    dims=["lat", "lon"],
    coords={"lat": lat_coords, "lon": lon_coords}
)
regridder = xe.Regridder(dummy_input, grid_out, method="bilinear", periodic=True)

# Loop over time and levelplv
for t in range(data.sizes["time"]):
    time_val = ds.time[t]
    regridded_levels = []
    
    for l in range(data.sizes["isobaricInhPa"]):
        lev_val = ds.isobaricInhPa[l]

        # Get 1D slice and reshapplve
        slice_1d = data.isel(time=t, isobaricInhPa=l).values
        slice_2d = slice_1d.reshape(nlat, nlon)

        da_in = xr.DataArray(
            slice_2d,
            dims=["lat", "lon"],
            coords={"lat": lat_coords, "lon": lon_coords}
        )

        # Regrid
        da_out = regridder(da_in)
        da_out = da_out.expand_dims(time=[time_val], isobaricInhPa=[lev_val])

        regridded_levels.append(da_out)
    
    regridded_time = xr.concat(regridded_levels, dim="isobaricInhPa")
    output_slices.append(regridded_time)

# Combine all time steps
ds_out = xr.concat(output_slices, dim="time")
ds_out.name = varname  # set name to original variable

In [ ]:
''' Interpolate to standard levels '''

# Surface pressure
sp = ds_sp['sp']  # [time, lat, lon]

# Variable to interpolate (e.g., temperature)
var = ds_in['avg_ttpm']  # [time, level, lat, lon]

# Hybrid coefficients
a = ds_in['hyam']  # shape: [level]
b = ds_in['hybm']  # shape: [level]

In [ ]:
# interpolate lat lon?